# 11 Silver AllergyIntolerance Clean

## Purpose

This notebook creates the Silver AllergyIntolerance table from raw FHIR AllergyIntolerance resources.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.allergy_intolerance_raw`
2. Extract allergy/intolerance fields
3. Flatten reaction information
4. Clean patient and encounter IDs
5. Convert timestamps
6. Save the clean table into the Silver layer

## Why We Are Doing This

AllergyIntolerance data is important for:
- patient safety analytics
- medication risk screening
- adverse reaction history
- clinical decision support
- downstream ML feature engineering

## Expected Final Output

A clean Delta table:

`healthcare_catalog.silver.allergy_intolerance_clean`

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
We need Spark functions to extract nested FHIR allergy fields and clean references.

### Expected Output
PySpark functions available.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze AllergyIntolerance Table

### What We Are Doing
We are reading the raw AllergyIntolerance table from the Bronze layer.

### Why We Are Doing This
Bronze contains raw nested FHIR AllergyIntolerance resources.

### Expected Output
A DataFrame named `allergy_raw_df`.

In [0]:
allergy_raw_df = spark.table(
    "healthcare_catalog.bronze.allergy_intolerance_raw"
)

print("Bronze allergy_intolerance_raw table loaded successfully.")

Bronze allergy_intolerance_raw table loaded successfully.


## Step 3 — Inspect Raw AllergyIntolerance Schema

### What We Are Doing
We are printing the schema.

### Why We Are Doing This
FHIR AllergyIntolerance resources are nested.  
We need to confirm exact fields before flattening.

### Expected Output
Schema fields such as:
- resource.id
- resource.patient.reference
- resource.encounter.reference
- resource.code.text
- resource.clinicalStatus
- resource.verificationStatus
- resource.category
- resource.criticality
- resource.reaction
- resource.recordedDate

In [0]:
allergy_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean AllergyIntolerance Columns

### What We Are Doing
We are extracting allergy and reaction-related fields from nested FHIR resources.

### Why We Are Doing This
Analytics-ready healthcare tables require flat structured columns.

### Fields We Will Extract

- allergy_id
- patient_reference
- encounter_reference
- allergy_description
- clinical_status
- verification_status
- allergy_category
- criticality
- reaction_manifestation
- reaction_severity
- recorded_datetime

### Expected Output
A DataFrame named `allergy_clean_df`.

In [0]:
allergy_clean_df = allergy_raw_df.select(

    col("resource.id").alias("allergy_id"),

    col("resource.patient.reference").alias("patient_reference"),

    col("resource.encounter.reference").alias("encounter_reference"),

    get_json_object(col("resource.code"), "$.text").alias("allergy_description"),

    col("resource.clinicalStatus.coding")[0]["code"].alias("clinical_status"),

    col("resource.verificationStatus.coding")[0]["code"].alias("verification_status"),

    col("resource.category")[0].alias("allergy_category"),

    col("resource.criticality").alias("criticality"),

    col("resource.reaction")[0]["manifestation"][0]["text"].alias("reaction_manifestation"),

    col("resource.reaction")[0]["severity"].alias("reaction_severity"),

    col("resource.recordedDate").alias("recorded_datetime")
)

print("Allergy clean DataFrame created successfully.")

Allergy clean DataFrame created successfully.


## Step 5 — Convert Allergy Recorded Date

### What We Are Doing
We are converting `recorded_datetime` into Spark timestamp format.

### Why We Are Doing This
Timestamp format supports:
- allergy history timeline
- patient safety analytics
- longitudinal patient records

### Expected Output
`recorded_datetime` becomes a timestamp column.

In [0]:
allergy_clean_df = allergy_clean_df.withColumn(
    "recorded_datetime",
    to_timestamp(col("recorded_datetime"))
)

print("Allergy recorded timestamp converted successfully.")

Allergy recorded timestamp converted successfully.


## Step 6 — Extract Clean Patient and Encounter IDs

### What We Are Doing
We are removing `urn:uuid:` from FHIR reference fields.

### Why We Are Doing This
Clean IDs are required for joining AllergyIntolerance with other Silver tables.

### Expected Output
New columns:
- patient_id

- encounter_id

In [0]:
allergy_clean_df = allergy_clean_df.withColumn(
    "patient_id",
    regexp_extract(col("patient_reference"), r"urn:uuid:(.*)", 1)
)

allergy_clean_df = allergy_clean_df.withColumn(
    "encounter_id",
    regexp_extract(col("encounter_reference"), r"urn:uuid:(.*)", 1)
)

print("Allergy patient and encounter IDs extracted successfully.")

Allergy patient and encounter IDs extracted successfully.


## Step 7 — Inspect Clean AllergyIntolerance Data

### What We Are Doing
We are displaying the clean AllergyIntolerance table.

### Why We Are Doing This
We need to verify:
- allergy descriptions
- reaction manifestation
- severity
- patient ID
- encounter ID
- timestamp

### Expected Output
A clean allergy-level healthcare table.

In [0]:
display(allergy_clean_df)

allergy_id,patient_reference,encounter_reference,allergy_description,clinical_status,verification_status,allergy_category,criticality,reaction_manifestation,reaction_severity,recorded_datetime,patient_id,encounter_id
7f020d47-51c8-740e-5233-1a6ef6484ddb,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,null,Eggs (edible) (substance),active,confirmed,food,low,null,null,1954-04-20T13:14:47.000Z,b0a06ead-cc42-aa48-dad6-841d4aa679fa,null
738f95d5-e2c3-c8e8-bdd9-61eba55a4ad6,urn:uuid:92fb7efc-5cfd-f8d3-927b-42f8ee099531,null,Fish (substance),active,confirmed,food,low,Dyspnea (finding),moderate,2014-12-14T17:56:06.000Z,92fb7efc-5cfd-f8d3-927b-42f8ee099531,null
5692c338-6acd-0a43-7a77-ca3e3cfbf370,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,Mold (organism),active,confirmed,environment,low,Allergic skin rash,mild,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null
44c35163-e3a9-05f1-aeda-4bb52e700a84,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,House dust mite (organism),active,confirmed,environment,low,null,null,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null
2efa474b-ffe8-6013-042c-8043c36f55a1,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,Animal dander (substance),active,confirmed,environment,low,Eruption of skin (disorder),mild,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null
f7f23555-9236-1bbd-75dd-316fd7d460cb,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,Grass pollen (substance),active,confirmed,environment,low,null,null,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null
527c7006-46d1-50ae-3b32-9dfedffcf794,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,Tree pollen (substance),active,confirmed,environment,low,null,null,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null
ddf60f5b-0701-c0eb-1ac1-220568f346d4,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,Eggs (edible) (substance),active,confirmed,food,low,Wheal (finding),moderate,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null
d6030079-58c7-b38a-ba55-db1e559ae598,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,Shellfish (substance),active,confirmed,food,low,Itching (finding),mild,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null
6db60d87-242e-28ce-d830-ccf69af0f95a,urn:uuid:346a1435-2455-914f-c287-7b88052d05db,null,Fish (substance),active,confirmed,food,low,Dyspnea (finding),moderate,1983-05-11T15:48:57.000Z,346a1435-2455-914f-c287-7b88052d05db,null


## Step 8 — Check Most Common Allergies

### What We Are Doing
We are counting allergy descriptions.

### Why We Are Doing This
This helps us understand allergy patterns in the dataset.

### Expected Output
Top allergy frequency table.

In [0]:
display(
    allergy_clean_df.groupBy(
        "allergy_description"
    ).count().orderBy(
        desc("count")
    )
)

allergy_description,count
Mold (organism),66
Animal dander (substance),64
House dust mite (organism),52
Grass pollen (substance),51
Tree pollen (substance),50
Latex (substance),25
Shellfish (substance),23
Peanut (substance),21
Eggs (edible) (substance),20
Bee venom (substance),18


## Step 9 — Check Reaction Severity Distribution

### What We Are Doing
We are counting reaction severity values.

### Why We Are Doing This
This supports patient safety and risk stratification analysis.

### Expected Output
Severity frequency table.

In [0]:
display(
    allergy_clean_df.groupBy(
        "reaction_severity"
    ).count().orderBy(
        desc("count")
    )
)

reaction_severity,count
null,253
mild,152
moderate,89
severe,5


## Step 10 — Check Null Values

### What We Are Doing
We are checking missing values in the clean AllergyIntolerance table.

### Why We Are Doing This
Silver-layer quality validation is required before Gold analytics.

### Expected Output
A null-count summary table.

In [0]:
display(
    allergy_clean_df.select(
        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)
            for column_name in allergy_clean_df.columns
        ]
    )
)

allergy_id,patient_reference,encounter_reference,allergy_description,clinical_status,verification_status,allergy_category,criticality,reaction_manifestation,reaction_severity,recorded_datetime,patient_id,encounter_id
0,0,499,0,0,0,0,0,253,253,0,0,499


## Step 11 — Save Silver AllergyIntolerance Table

### What We Are Doing
We are saving the clean AllergyIntolerance table into the Silver layer.

### Why We Are Doing This
This creates a reusable Delta table for analytics, dashboards, and ML.

### Expected Output
A Delta table:

`healthcare_catalog.silver.allergy_intolerance_clean`

In [0]:
allergy_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.allergy_intolerance_clean")

print("Silver allergy_intolerance_clean table saved successfully.")

Silver allergy_intolerance_clean table saved successfully.


## Step 12 — Verify Silver Tables

### What We Are Doing
We are listing all Silver tables.

### Why We Are Doing This
We want to confirm that `allergy_intolerance_clean` was saved successfully.

### Expected Output
`allergy_intolerance_clean` should appear in the Silver table list.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|silver  |allergy_intolerance_clean|false      |
|silver  |careplan_clean           |false      |
|silver  |condition_clean          |false      |
|silver  |encounter_clean          |false      |
|silver  |immunization_clean       |false      |
|silver  |medication_request_clean |false      |
|silver  |observation_clean        |false      |
|silver  |patient_clean            |false      |
|silver  |procedure_clean          |false      |
+--------+-------------------------+-----------+

